# 02｜狀態與共用工具

建立 Flask 應用程式、執行狀態、跨平台檔案路徑與影片工具函式。


In [ ]:
from pathlib import Path
import csv
import io
import os
import threading
import time
from collections import defaultdict, deque
from datetime import datetime

import cv2
import numpy as np
from flask import Flask, Response, jsonify, render_template_string, request, send_file
from linebot import LineBotApi
from linebot.models import TextSendMessage

TARGET_CLASSES = ["Drowning", "Person out of water", "Swimming"]
DISPLAY_MAX_WIDTH = 720
WORK_DIR = Path.cwd()
UPLOAD_PATH = str(WORK_DIR / "uploaded_video.mp4")
LOG_PATH = str(WORK_DIR / "detection_log.csv")

app = Flask(__name__)

_black_frame = np.zeros((360, 640, 3), dtype=np.uint8)
cv2.putText(_black_frame, "No video loaded", (170, 180),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (150, 150, 150), 2)
_, _black_buffer = cv2.imencode(".jpg", _black_frame)
BLACK_FRAME_JPEG = _black_buffer.tobytes()


class DetectionState:
    def __init__(self):
        self.lock = threading.Lock()
        self.latest_frame_jpeg = None
        self.stats_text = "尚未開始處理"
        self.warning_active = False
        self.running = False
        self.paused = False
        self.thread = None
        self.file_uploaded = False
        self.log_ready = False

        self.line_enabled = False
        self.line_token = ""

    def update_frame(self, jpeg_bytes):
        with self.lock:
            self.latest_frame_jpeg = jpeg_bytes

    def clear_frame(self):
        with self.lock:
            self.latest_frame_jpeg = None

    def update_stats(self, text, warning):
        with self.lock:
            self.stats_text = text
            self.warning_active = warning

    def get_frame(self):
        with self.lock:
            return self.latest_frame_jpeg

    def get_stats(self):
        with self.lock:
            return self.stats_text, self.warning_active, self.paused, self.running, self.log_ready

state = DetectionState()

def fmt_time(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"

def extract_first_frame_jpeg(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None

    h, w = frame.shape[:2]
    if w > DISPLAY_MAX_WIDTH:
        scale = DISPLAY_MAX_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    ok, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return buffer.tobytes() if ok else None

def resolve_source(raw_source, is_stream):
    if not is_stream:
        return raw_source
    raw_source = raw_source.strip()
    if raw_source.isdigit():
        return int(raw_source)
    return raw_source
